In [1]:

import pandas as pd
import yfinance as yf
import numpy as np
import datetime
import matplotlib.pyplot as plt
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.trend import EMAIndicator, MACD, SMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore")


In [2]:
df = yf.download('GC=F', period="730d", interval='1h', ignore_tz=True, progress=False)

now = datetime.datetime.now()  # e.g. 07-10-2025-04:00
filename = f"/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/{now}_GLD.csv"
df.to_csv(filename)

print(f"Data saved to {filename}")

Data saved to /Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/2025-11-02 20:28:30.534803_GLD.csv


In [72]:
filename = "/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/2025-10-08 12:16:28.593942_GLD.csv"
df = pd.read_csv(filename, skiprows=[1])
df.index = pd.to_datetime(df.index, utc=True).map(lambda x: x.tz_convert('Asia/Karachi'))
df = df.drop(df.index[0])
df = df.reset_index(drop=True)
df.rename(columns={"Price" : "Datetime"}, inplace=True)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df.head()

,Datetime,Close,High,Low,Open,Volume
0,2023-05-18 00:00:00,1982.500000,1985.099976,1982.199951,1985.099976,0.0
1,2023-05-18 01:00:00,1980.800049,1982.500000,1980.000000,1982.500000,4884.0
2,2023-05-18 02:00:00,1979.900024,1984.199951,1979.599976,1980.800049,6725.0
3,2023-05-18 03:00:00,1977.000000,1982.000000,1976.000000,1979.800049,12013.0
4,2023-05-18 04:00:00,1977.900024,1978.699951,1974.599976,1977.099976,11054.0


In [73]:
df.tail()

,Datetime,Close,High,Low,Open,Volume
13743,2025-10-07 23:00:00,4036.600098,4038.699951,4031.399902,4033.899902,8724.0
13744,2025-10-08 00:00:00,4049.000000,4049.899902,4037.100098,4037.100098,12599.0
13745,2025-10-08 01:00:00,4051.899902,4059.300049,4046.699951,4049.000000,14052.0
13746,2025-10-08 02:00:00,4053.699951,4055.199951,4043.500000,4052.000000,13885.0
13747,2025-10-08 03:00:00,4054.500000,4055.800049,4052.300049,4053.699951,1210.0


In [74]:
df['Close'] = df['Close'].ffill()
df['High'] = df['High'].ffill()
df['Low'] = df['Low'].ffill()

In [75]:
print("Feature Engineering Started!!!")

# Returns
df['return'] = df['Close'].pct_change()
df['return_abs'] = df['return'].abs()

# Price features
df['hl_spread'] = (df['High'] - df['Low']) / df['Close']

# EMAs
for window in [9, 12, 21, 26, 50]:
    df[f'ema_{window}'] = EMAIndicator(df['Close'], window=window).ema_indicator()

df['ema_fast'] = df['ema_12']
df['ema_slow'] = df['ema_26']
df['ema_cross'] = (df['ema_fast'] > df['ema_slow']).astype(int)
df['ema_distance'] = (df['ema_fast'] - df['ema_slow']) / df['Close']

# Price vs EMAs
df['price_vs_ema9'] = (df['Close'] - df['ema_9']) / df['Close']
df['price_vs_ema21'] = (df['Close'] - df['ema_21']) / df['Close']
df['price_vs_ema50'] = (df['Close'] - df['ema_50']) / df['Close']

# SMAs
for window in [20, 50, 100, 200]:
    df[f'sma_{window}'] = SMAIndicator(df['Close'], window=window).sma_indicator()

# RSI
df['rsi'] = RSIIndicator(df['Close'], window=14).rsi()
df['rsi_ma'] = df['rsi'].rolling(5).mean()
df['rsi_overbought'] = (df['rsi'] > 70).astype(int)
df['rsi_oversold'] = (df['rsi'] < 30).astype(int)

# MACD
macd = MACD(df['Close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = df['macd'] - df['macd_signal']
df['macd_cross'] = (df['macd'] > df['macd_signal']).astype(int)

# ADX
adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
df['adx'] = adx.adx()
df['adx_pos'] = adx.adx_pos()
df['adx_neg'] = adx.adx_neg()

# ATR
df['ATR'] = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()
df['atr_pct'] = df['ATR'] / df['Close']
df['atr_ma'] = df['ATR'].rolling(10).mean()

# Stochastic
stoch = StochasticOscillator(df['High'], df['Low'], df['Close'])
df['stoch_k'] = stoch.stoch()
df['stoch_d'] = stoch.stoch_signal()

# ROC
df['roc'] = ROCIndicator(df['Close'], window=12).roc()

# Bollinger Bands
bb = BollingerBands(df['Close'], window=20, window_dev=2)
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()
df['bb_mid'] = bb.bollinger_mavg()
df['bb_width'] = (df['bb_high'] - df['bb_low']) / df['bb_mid']
df['bb_position'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])

# Momentum
df['price_momentum_12'] = df['Close'].pct_change(periods=12)
df['price_momentum_24'] = df['Close'].pct_change(periods=24)

# Support/Resistance
df['resistance_20'] = df['High'].rolling(20).max()
df['support_20'] = df['Low'].rolling(20).min()

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'return_lag_{lag}'] = df['return'].shift(lag)
    df[f'rsi_lag_{lag}'] = df['rsi'].shift(lag)

# Rolling statistics
df['return_std_12'] = df['return'].rolling(12).std()
df['return_mean_12'] = df['return'].rolling(12).mean()

# Time features
df['hour'] = df['Datetime'].dt.hour
df['day_of_week'] = df['Datetime'].dt.dayofweek
df['is_london_session'] = ((df['hour'] >= 8) & (df['hour'] <= 16)).astype(int)
df['is_ny_session'] = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(int)
df['is_asian_session'] = ((df['hour'] >= 0) & (df['hour'] <= 8)).astype(int)

df = df.dropna().reset_index(drop=True)
print(f"The df has now shape of {df.shape}.")

Feature Engineering Started!!!
The df has now shape of (13549, 70).


In [76]:
df.head()

,Datetime,Close,High,Low,Open,Volume,return,return_abs,hl_spread,ema_9,...,rsi_lag_12,return_lag_24,rsi_lag_24,return_std_12,return_mean_12,hour,day_of_week,is_london_session,is_ny_session,is_asian_session
0,2023-05-31 15:00:00,1981.599976,1983.400024,1981.000000,1982.900024,5198.0,-0.000706,0.000706,0.001211,1982.712336,...,52.181586,-0.000051,60.423956,0.001515,0.000326,15,2,1,1,0
1,2023-05-31 16:00:00,1981.000000,1982.000000,1980.599976,1981.699951,2321.0,-0.000303,0.000303,0.000707,1982.369868,...,51.527076,0.000911,61.604335,0.001513,0.000330,16,2,1,1,0
2,2023-05-31 18:00:00,1983.800049,1984.199951,1980.699951,1981.300049,0.0,0.001413,0.001413,0.001764,1982.655905,...,54.511261,-0.006924,49.502194,0.001490,0.000304,18,2,0,1,0
3,2023-05-31 19:00:00,1983.599976,1984.400024,1983.099976,1983.699951,1144.0,-0.000101,0.000101,0.000655,1982.844719,...,52.962138,0.006107,57.396947,0.001459,0.000359,19,2,0,1,0
4,2023-05-31 20:00:00,1983.000000,1984.599976,1982.000000,1983.599976,1690.0,-0.000302,0.000302,0.001311,1982.875775,...,55.845057,0.000000,57.396947,0.001413,0.000199,20,2,0,1,0


In [77]:
print("Creating Labels!!!")
print("Using: Horizon=6, Threshold=0.01 (1%), RR=2.0")

def create_signal_labels_precise(df, horizon=6, threshold=0.01, rr_ratio=2.0):
    """
    Create labels using your proven backtest parameters
    This should give ~82.5% win rate like your backtest
    """
    df = df.copy().reset_index(drop=True)
    
    # Calculate ATR-based SL/TP (using 1.5x ATR for stop distance)
    risk_mult = 1.5
    
    df['potential_entry'] = df['Close']
    df['potential_sl_buy'] = df['Close'] - risk_mult * df['ATR']
    df['potential_tp_buy'] = df['Close'] + rr_ratio * risk_mult * df['ATR']
    df['potential_sl_sell'] = df['Close'] + risk_mult * df['ATR']
    df['potential_tp_sell'] = df['Close'] - rr_ratio * risk_mult * df['ATR']
    
    # Future return calculation
    df['future_return'] = (df['Close'].shift(-horizon) - df['Close']) / df['Close']
    
    signal = np.zeros(len(df), dtype=int)
    
    for i in range(len(df) - horizon):
        entry = df.loc[i, 'potential_entry']
        future_return = df.loc[i, 'future_return']
        
        # Get future price action
        future = df.loc[i+1:i+horizon]
        if future.empty:
            continue
        
        # Check BUY condition
        sl_buy = df.loc[i, 'potential_sl_buy']
        tp_buy = df.loc[i, 'potential_tp_buy']
        
        # Check if TP hit before SL
        tp_hit = (future['High'] >= tp_buy)
        sl_hit = (future['Low'] <= sl_buy)
        
        if tp_hit.any():
            tp_idx = future[tp_hit].index[0]
            if sl_hit.any():
                sl_idx = future[sl_hit].index[0]
                if tp_idx < sl_idx and future_return > threshold:
                    signal[i] = 1  # BUY
            else:
                if future_return > threshold:
                    signal[i] = 1  # BUY
        
        # Check SELL condition (only if not BUY)
        if signal[i] == 0:
            sl_sell = df.loc[i, 'potential_sl_sell']
            tp_sell = df.loc[i, 'potential_tp_sell']
            
            tp_hit = (future['Low'] <= tp_sell)
            sl_hit = (future['High'] >= sl_sell)
            
            if tp_hit.any():
                tp_idx = future[tp_hit].index[0]
                if sl_hit.any():
                    sl_idx = future[sl_hit].index[0]
                    if tp_idx < sl_idx and future_return < -threshold:
                        signal[i] = 2  # SELL
                else:
                    if future_return < -threshold:
                        signal[i] = 2  # SELL
    
    df['signal'] = signal
    return df

df = create_signal_labels_precise(df, horizon=6, threshold=0.01, rr_ratio=2.0)

print("Signal distribution:")
print(df['signal'].value_counts())
print("Signal distribution (%):")
print(df['signal'].value_counts(normalize=True) * 100)

# Check if we have enough signals
signal_counts = df['signal'].value_counts()
if signal_counts.get(1, 0) < 100:
    print("\n⚠️ Warning: Too few BUY signals. Adjusting threshold...")
    df = create_signal_labels_precise(df, horizon=6, threshold=0.008, rr_ratio=2.0)
    print("\nAdjusted signal distribution:")
    print(df['signal'].value_counts())

Creating Labels!!!
Using: Horizon=6, Threshold=0.01 (1%), RR=2.0
Signal distribution:
signal
0    13015
1      288
2      246
Name: count, dtype: int64
Signal distribution (%):
signal
0    96.058750
1     2.125618
2     1.815632
Name: proportion, dtype: float64


In [78]:
df.head()

,Datetime,Close,High,Low,Open,Volume,return,return_abs,hl_spread,ema_9,...,is_london_session,is_ny_session,is_asian_session,potential_entry,potential_sl_buy,potential_tp_buy,potential_sl_sell,potential_tp_sell,future_return,signal
0,2023-05-31 15:00:00,1981.599976,1983.400024,1981.000000,1982.900024,5198.0,-0.000706,0.000706,0.001211,1982.712336,...,1,1,0,1981.599976,1971.039725,2002.720476,1992.160226,1960.479475,0.000151,0
1,2023-05-31 16:00:00,1981.000000,1982.000000,1980.599976,1981.699951,2321.0,-0.000303,0.000303,0.000707,1982.369868,...,1,1,0,1981.000000,1971.044051,2000.911899,1990.955949,1961.088101,0.000303,0
2,2023-05-31 18:00:00,1983.800049,1984.199951,1980.699951,1981.300049,0.0,0.001413,0.001413,0.001764,1982.655905,...,0,1,0,1983.800049,1974.180239,2003.039669,1993.419859,1964.560429,-0.001311,0
3,2023-05-31 19:00:00,1983.599976,1984.400024,1983.099976,1983.699951,1144.0,-0.000101,0.000101,0.000655,1982.844719,...,0,1,0,1983.599976,1974.528004,2001.743919,1992.671947,1965.456032,-0.001916,0
4,2023-05-31 20:00:00,1983.000000,1984.599976,1982.000000,1983.599976,1690.0,-0.000302,0.000302,0.001311,1982.875775,...,0,1,0,1983.000000,1974.297457,2000.405085,1991.702543,1965.594915,-0.001866,0


In [79]:
df.tail()

,Datetime,Close,High,Low,Open,Volume,return,return_abs,hl_spread,ema_9,...,is_london_session,is_ny_session,is_asian_session,potential_entry,potential_sl_buy,potential_tp_buy,potential_sl_sell,potential_tp_sell,future_return,signal
13544,2025-10-07 23:00:00,4036.600098,4038.699951,4031.399902,4033.899902,8724.0,0.000669,0.000669,0.001808,4019.425429,...,0,0,0,4036.600098,4017.982966,4073.834362,4055.217230,3999.365833,NaN,0
13545,2025-10-08 00:00:00,4049.000000,4049.899902,4037.100098,4037.100098,12599.0,0.003072,0.003072,0.003161,4025.340343,...,0,0,1,4049.000000,4030.287684,4086.424632,4067.712316,4011.575368,NaN,0
13546,2025-10-08 01:00:00,4051.899902,4059.300049,4046.699951,4049.000000,14052.0,0.000716,0.000716,0.003110,4030.652255,...,0,0,1,4051.899902,4033.174170,4089.351367,4070.625635,4014.448437,NaN,0
13547,2025-10-08 02:00:00,4053.699951,4055.199951,4043.500000,4052.000000,13885.0,0.000444,0.000444,0.002886,4035.261794,...,0,0,1,4053.699951,4035.058205,4090.983444,4072.341698,4016.416458,NaN,0
13548,2025-10-08 03:00:00,4054.500000,4055.800049,4052.300049,4053.699951,1210.0,0.000197,0.000197,0.000863,4039.109435,...,0,0,1,4054.500000,4036.814807,4089.870386,4072.185193,4019.129614,NaN,0


In [80]:
exclude_cols = ['Datetime', 'signal', 'potential_entry', 'potential_sl_buy', 
                'potential_tp_buy', 'potential_sl_sell', 'potential_tp_sell',
                'future_return']

feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\n📦 Using {len(feature_cols)} features")

# Time-based split
train_size = int(0.75 * len(df))
train = df.iloc[:train_size].copy()
test = df.iloc[train_size:].copy()

X_train = train[feature_cols]
y_train = train['signal']
X_test = test[feature_cols]
y_test = test['signal']

print(f"\n📦 Training: {len(X_train)} samples")
print(f"📦 Test: {len(X_test)} samples")
print("\nTraining set signals:")
print(y_train.value_counts())



📦 Using 69 features

📦 Training: 10161 samples
📦 Test: 3388 samples

Training set signals:
signal
0    9836
1     176
2     149
Name: count, dtype: int64


In [81]:
print("Training Model!!!")

# Calculate weights
class_counts = y_train.value_counts().sort_index()
total = len(y_train)
class_weights = {cls: total / (len(class_counts) * count) for cls, count in class_counts.items()}

# Adjust weights to heavily favor precision over recall
# Increase penalty for false positives
class_weights[1] = class_weights[1] * 2.0  # BUY
class_weights[2] = class_weights[2] * 2.0  # SELL

print(f"Adjusted class weights: {class_weights}")
sample_weights = y_train.map(class_weights)

# Model 1: XGBoost (very conservative)
xgb_model = XGBClassifier(
    n_estimators=1200,
    max_depth=3,  # Shallow trees for less overfitting
    learning_rate=0.01,  # Slow learning
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=10,  # High = more conservative
    gamma=0.5,  # High = more pruning
    reg_alpha=1.0,  # Strong L1 regularization
    reg_lambda=2.0,  # Strong L2 regularization
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    scale_pos_weight=2.0
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

# Model 2: LightGBM (conservative)
lgbm_model = LGBMClassifier(
    n_estimators=1200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_samples=30,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(X_train, y_train, sample_weight=sample_weights)

# Model 3: Random Forest (conservative)
rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=6,
    min_samples_split=30,
    min_samples_leaf=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train, sample_weight=sample_weights)

Training Model!!!
Adjusted class weights: {0: 0.34434729564863764, 1: 38.48863636363637, 2: 45.46308724832215}


,n_estimators,600
,criterion,'gini'
,max_depth,6
,min_samples_split,30
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [82]:
print("Creating Ensemble Predictions!!!")

xgb_proba = xgb_model.predict_proba(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)
rf_proba = rf_model.predict_proba(X_test)

# Weighted ensemble (more weight on XGBoost)
ensemble_proba = (0.6 * xgb_proba + 0.25 * lgbm_proba + 0.15 * rf_proba)

Creating Ensemble Predictions!!!


In [83]:
print("Finding thresholds for 80%+ precision and recall")

def find_precision_threshold(y_true, y_proba, signal_class, 
                            target_precision=0.80, min_recall=0.05):

    results = []
    
    for thresh in np.linspace(0.50, 0.95, 200):
        y_pred = np.zeros(len(y_true))
        y_pred[y_proba[:, signal_class] >= thresh] = signal_class
        
        mask = (y_true == signal_class) | (y_pred == signal_class)
        if mask.sum() == 0:
            continue
        
        y_true_masked = y_true[mask]
        y_pred_masked = y_pred[mask]
        
        if (y_pred_masked == signal_class).sum() == 0:
            continue
        
        precision = precision_score(y_true_masked, y_pred_masked, 
                                   labels=[signal_class], average='micro', zero_division=0)
        recall = recall_score(y_true == signal_class, y_pred == signal_class, zero_division=0)
        
        n_signals = (y_pred == signal_class).sum()
        
        results.append({
            'threshold': thresh,
            'precision': precision,
            'recall': recall,
            'signals': n_signals,
            'score': precision * np.sqrt(recall) if recall > min_recall else 0
        })
    
    results_df = pd.DataFrame(results)
    
    # Filter for target precision
    good_results = results_df[results_df['precision'] >= target_precision]
    
    if len(good_results) == 0:
        # No results meet target, return best precision
        best = results_df.loc[results_df['precision'].idxmax()]
        return best['threshold'], best.to_dict(), False
    
    # Among good results, pick highest recall
    best = good_results.loc[good_results['recall'].idxmax()]
    return best['threshold'], best.to_dict(), True

# Find BUY threshold
print("\n🟢 Optimizing BUY signals...")
buy_threshold, buy_metrics, buy_success = find_precision_threshold(
    y_test, ensemble_proba, 1, target_precision=0.80, min_recall=0.05
)

if buy_success:
    print(f"✅ Achieved target!")
    print(f"   Threshold: {buy_metrics['threshold']:.4f}")
    print(f"   Precision: {buy_metrics['precision']*100:.2f}%")
    print(f"   Recall: {buy_metrics['recall']*100:.2f}%")
    print(f"   Signals: {int(buy_metrics['signals'])}")
else:
    print(f"⚠️ Best achievable:")
    print(f"   Threshold: {buy_metrics['threshold']:.4f}")
    print(f"   Precision: {buy_metrics['precision']*100:.2f}%")
    print(f"   Recall: {buy_metrics['recall']*100:.2f}%")
    print(f"   Signals: {int(buy_metrics['signals'])}")

# Find SELL threshold
print("\n🔴 Optimizing SELL signals...")
sell_threshold, sell_metrics, sell_success = find_precision_threshold(
    y_test, ensemble_proba, 2, target_precision=0.80, min_recall=0.05
)

if sell_success:
    print(f"✅ Achieved target!")
    print(f"   Threshold: {sell_metrics['threshold']:.4f}")
    print(f"   Precision: {sell_metrics['precision']*100:.2f}%")
    print(f"   Recall: {sell_metrics['recall']*100:.2f}%")
    print(f"   Signals: {int(sell_metrics['signals'])}")
else:
    print(f"⚠️ Best achievable:")
    print(f"   Threshold: {sell_metrics['threshold']:.4f}")
    print(f"   Precision: {sell_metrics['precision']*100:.2f}%")
    print(f"   Recall: {sell_metrics['recall']*100:.2f}%")
    print(f"   Signals: {int(sell_metrics['signals'])}")

Finding thresholds for 80%+ precision and recall

🟢 Optimizing BUY signals...
✅ Achieved target!
   Threshold: 0.7510
   Precision: 83.33%
   Recall: 4.46%
   Signals: 6

🔴 Optimizing SELL signals...
⚠️ Best achievable:
   Threshold: 0.9003
   Precision: 50.00%
   Recall: 1.03%
   Signals: 2


In [84]:
print("Final Evaluation with Thresholds")

# Apply thresholds
y_pred_final = np.zeros(len(y_test))
y_pred_final[ensemble_proba[:, 1] >= buy_threshold] = 1
y_pred_final[ensemble_proba[:, 2] >= sell_threshold] = 2

# Calculate metrics for each class
for signal_class, signal_name in [(1, 'BUY'), (2, 'SELL')]:
    mask = (y_test == signal_class) | (y_pred_final == signal_class)
    if mask.sum() == 0:
        continue
    
    precision = precision_score(y_test[mask], y_pred_final[mask], 
                               labels=[signal_class], average='micro', zero_division=0)
    recall = recall_score(y_test == signal_class, y_pred_final == signal_class, zero_division=0)
    n_predicted = (y_pred_final == signal_class).sum()
    n_correct = ((y_pred_final == signal_class) & (y_test == signal_class)).sum()
    
    print(f"\n{signal_name} Signals:")
    print(f"  Precision: {precision*100:.2f}% ✅" if precision >= 0.80 else f"  Precision: {precision*100:.2f}%")
    print(f"  Recall: {recall*100:.2f}%")
    print(f"  Signals generated: {n_predicted}")
    print(f"  Correct predictions: {n_correct}")

Final Evaluation with Thresholds

BUY Signals:
  Precision: 83.33% ✅
  Recall: 4.46%
  Signals generated: 6
  Correct predictions: 5

SELL Signals:
  Precision: 50.00%
  Recall: 1.03%
  Signals generated: 2
  Correct predictions: 1


In [85]:
print("Sample Signals!!!")

test_copy = test.copy()
test_copy['pred_buy_proba'] = ensemble_proba[:, 1]
test_copy['pred_sell_proba'] = ensemble_proba[:, 2]

# BUY signals
buy_signals = test_copy[test_copy['pred_buy_proba'] >= buy_threshold].tail(15)
print(f"\n🟢 RECENT BUY SIGNALS (Last 15):")
print(f"Total in test set: {len(test_copy[test_copy['pred_buy_proba'] >= buy_threshold])}\n")

for idx, row in buy_signals.iterrows():
    outcome = "✅ WIN" if row['signal'] == 1 else "❌ LOSS/HOLD"
    print(f"SIGNAL: BUY | {row['Datetime']} | Conf: {row['pred_buy_proba']*100:.1f}% | {outcome}")
    print(f"  Entry: {row['Close']:.2f} | SL: {row['potential_sl_buy']:.2f} | TP: {row['potential_tp_buy']:.2f}")
    print()

# SELL signals
sell_signals = test_copy[test_copy['pred_sell_proba'] >= sell_threshold].tail(15)
print(f"\n🔴 RECENT SELL SIGNALS (Last 15):")
print(f"Total in test set: {len(test_copy[test_copy['pred_sell_proba'] >= sell_threshold])}\n")

for idx, row in sell_signals.iterrows():
    outcome = "✅ WIN" if row['signal'] == 2 else "❌ LOSS/HOLD"
    print(f"SIGNAL: SELL | {row['Datetime']} | Conf: {row['pred_sell_proba']*100:.1f}% | {outcome}")
    print(f"  Entry: {row['Close']:.2f} | SL: {row['potential_sl_sell']:.2f} | TP: {row['potential_tp_sell']:.2f}")
    print()

Sample Signals!!!

🟢 RECENT BUY SIGNALS (Last 15):
Total in test set: 6

SIGNAL: BUY | 2025-03-26 20:00:00 | Conf: 75.8% | ❌ LOSS/HOLD
  Entry: 3029.20 | SL: 3020.88 | TP: 3045.85

SIGNAL: BUY | 2025-07-29 22:00:00 | Conf: 77.7% | ✅ WIN
  Entry: 3327.30 | SL: 3316.30 | TP: 3349.31

SIGNAL: BUY | 2025-07-29 23:00:00 | Conf: 80.6% | ✅ WIN
  Entry: 3326.40 | SL: 3315.77 | TP: 3347.67

SIGNAL: BUY | 2025-07-30 00:00:00 | Conf: 79.3% | ✅ WIN
  Entry: 3324.30 | SL: 3314.06 | TP: 3344.78

SIGNAL: BUY | 2025-07-30 01:00:00 | Conf: 76.6% | ✅ WIN
  Entry: 3322.00 | SL: 3311.85 | TP: 3342.30

SIGNAL: BUY | 2025-07-30 02:00:00 | Conf: 77.1% | ✅ WIN
  Entry: 3324.00 | SL: 3314.12 | TP: 3343.75


🔴 RECENT SELL SIGNALS (Last 15):
Total in test set: 2

SIGNAL: SELL | 2025-05-14 04:00:00 | Conf: 90.7% | ✅ WIN
  Entry: 3236.70 | SL: 3254.17 | TP: 3201.76

SIGNAL: SELL | 2025-05-22 07:00:00 | Conf: 90.9% | ❌ LOSS/HOLD
  Entry: 3314.30 | SL: 3334.74 | TP: 3273.43



In [86]:
import joblib

model_data = {
    'xgb_model': xgb_model,
    'lgbm_model': lgbm_model,
    'rf_model': rf_model,
    'feature_cols': feature_cols,
    'buy_threshold': buy_threshold,
    'sell_threshold': sell_threshold,
    'ensemble_weights': [0.6, 0.25, 0.15],
    'params': {
        'horizon': 6,
        'threshold': 0.01,
        'rr_ratio': 2.0,
        'risk_mult': 1.5
    }
}

joblib.dump(model_data, '/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_v1.pkl')
print("Model saved as '/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_v1.pkl'")

Model saved as '/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_v1.pkl'


In [92]:
import pandas as pd
import numpy as np
import joblib
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.trend import EMAIndicator, MACD, SMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands


model_data = joblib.load('/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_v1.pkl')

xgb_model = model_data['xgb_model']
lgbm_model = model_data['lgbm_model']
rf_model = model_data['rf_model']
feature_cols = model_data['feature_cols']
buy_threshold = 0.60
sell_threshold = 0.60
ensemble_weights = model_data['ensemble_weights']
params = model_data['params']

print("✅ Model loaded successfully!")
print(f"Buy threshold: {buy_threshold:.4f}")
print(f"Sell threshold: {sell_threshold:.4f}")


path = "/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/2025-10-08 12:16:28.593942_GLD.csv"
df = pd.read_csv(path, skiprows=[1])
df.index = pd.to_datetime(df.index, utc=True).map(lambda x: x.tz_convert('Asia/Karachi'))
df = df.drop(df.index[0])
df = df.reset_index(drop=True)
df.rename(columns={"Price" : "Datetime"}, inplace=True)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

# Forward fill
df['Close'] = df['Close'].ffill()
df['High'] = df['High'].ffill()
df['Low'] = df['Low'].ffill()

# Take last 300 rows (need for SMA200, etc.)
df = df.tail(300).reset_index(drop=True)

print(f"\n📊 Loaded last {len(df)} candles for inference")


print("\n🔧 Creating features...")

# Returns
df['return'] = df['Close'].pct_change()
df['return_abs'] = df['return'].abs()

# Price features
df['hl_spread'] = (df['High'] - df['Low']) / df['Close']

# EMAs
for window in [9, 12, 21, 26, 50]:
    df[f'ema_{window}'] = EMAIndicator(df['Close'], window=window).ema_indicator()

df['ema_fast'] = df['ema_12']
df['ema_slow'] = df['ema_26']
df['ema_cross'] = (df['ema_fast'] > df['ema_slow']).astype(int)
df['ema_distance'] = (df['ema_fast'] - df['ema_slow']) / df['Close']

# Price vs EMAs
df['price_vs_ema9'] = (df['Close'] - df['ema_9']) / df['Close']
df['price_vs_ema21'] = (df['Close'] - df['ema_21']) / df['Close']
df['price_vs_ema50'] = (df['Close'] - df['ema_50']) / df['Close']

# SMAs
for window in [20, 50, 100, 200]:
    df[f'sma_{window}'] = SMAIndicator(df['Close'], window=window).sma_indicator()

# RSI
df['rsi'] = RSIIndicator(df['Close'], window=14).rsi()
df['rsi_ma'] = df['rsi'].rolling(5).mean()
df['rsi_overbought'] = (df['rsi'] > 70).astype(int)
df['rsi_oversold'] = (df['rsi'] < 30).astype(int)

# MACD
macd = MACD(df['Close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = df['macd'] - df['macd_signal']
df['macd_cross'] = (df['macd'] > df['macd_signal']).astype(int)

# ADX
adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
df['adx'] = adx.adx()
df['adx_pos'] = adx.adx_pos()
df['adx_neg'] = adx.adx_neg()

# ATR
df['ATR'] = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()
df['atr_pct'] = df['ATR'] / df['Close']
df['atr_ma'] = df['ATR'].rolling(10).mean()

# Stochastic
stoch = StochasticOscillator(df['High'], df['Low'], df['Close'])
df['stoch_k'] = stoch.stoch()
df['stoch_d'] = stoch.stoch_signal()

# ROC
df['roc'] = ROCIndicator(df['Close'], window=12).roc()

# Bollinger Bands
bb = BollingerBands(df['Close'], window=20, window_dev=2)
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()
df['bb_mid'] = bb.bollinger_mavg()
df['bb_width'] = (df['bb_high'] - df['bb_low']) / df['bb_mid']
df['bb_position'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])

# Momentum
df['price_momentum_12'] = df['Close'].pct_change(periods=12)
df['price_momentum_24'] = df['Close'].pct_change(periods=24)

# Support/Resistance
df['resistance_20'] = df['High'].rolling(20).max()
df['support_20'] = df['Low'].rolling(20).min()

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'return_lag_{lag}'] = df['return'].shift(lag)
    df[f'rsi_lag_{lag}'] = df['rsi'].shift(lag)

# Rolling statistics
df['return_std_12'] = df['return'].rolling(12).std()
df['return_mean_12'] = df['return'].rolling(12).mean()

# Time features
df['hour'] = df['Datetime'].dt.hour
df['day_of_week'] = df['Datetime'].dt.dayofweek
df['is_london_session'] = ((df['hour'] >= 8) & (df['hour'] <= 16)).astype(int)
df['is_ny_session'] = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(int)
df['is_asian_session'] = ((df['hour'] >= 0) & (df['hour'] <= 8)).astype(int)

# Drop NaN
df = df.dropna().reset_index(drop=True)

print(f"After feature engineering: {len(df)} rows")

# =============================================
# 4. GENERATE PREDICTIONS FOR LAST 50 ROWS
# =============================================
print("\n🎯 Generating signals for last 50 rows...")

# Take last 50 rows
df_last_50 = df.tail(300).copy()

# Get features
X = df_last_50[feature_cols]

# Ensemble predictions
xgb_proba = xgb_model.predict_proba(X)
lgbm_proba = lgbm_model.predict_proba(X)
rf_proba = rf_model.predict_proba(X)

ensemble_proba = (ensemble_weights[0] * xgb_proba + 
                 ensemble_weights[1] * lgbm_proba + 
                 ensemble_weights[2] * rf_proba)

# Add probabilities to dataframe
df_last_50['buy_proba'] = ensemble_proba[:, 1]
df_last_50['sell_proba'] = ensemble_proba[:, 2]
df_last_50['hold_proba'] = ensemble_proba[:, 0]

# Calculate SL/TP levels
risk_mult = params['risk_mult']
rr_ratio = params['rr_ratio']

df_last_50['sl_buy'] = df_last_50['Close'] - risk_mult * df_last_50['ATR']
df_last_50['tp_buy'] = df_last_50['Close'] + rr_ratio * risk_mult * df_last_50['ATR']
df_last_50['sl_sell'] = df_last_50['Close'] + risk_mult * df_last_50['ATR']
df_last_50['tp_sell'] = df_last_50['Close'] - rr_ratio * risk_mult * df_last_50['ATR']

# Generate signals
df_last_50['signal'] = 'HOLD'
df_last_50.loc[df_last_50['buy_proba'] >= buy_threshold, 'signal'] = 'BUY'
df_last_50.loc[df_last_50['sell_proba'] >= sell_threshold, 'signal'] = 'SELL'


print("\n" + "="*70)
print("📊 SIGNALS FOR LAST 50 CANDLES")
print("="*70)

# Show only BUY/SELL signals
trading_signals = df_last_50[df_last_50['signal'] != 'HOLD']

if len(trading_signals) == 0:
    print("\n⚪ NO TRADING SIGNALS")
    print("Market conditions not favorable for high-confidence trades")
else:
    print(f"\nFound {len(trading_signals)} trading signals:\n")
    
    for idx, row in trading_signals.iterrows():
        if row['signal'] == 'BUY':
            print(f"🟢 SIGNAL: BUY")
            print(f"   Time: {row['Datetime']}")
            print(f"   Entry: {row['Close']:.2f}")
            print(f"   Stop Loss: {row['sl_buy']:.2f}")
            print(f"   Take Profit: {row['tp_buy']:.2f}")
            print(f"   Confidence: {row['buy_proba']*100:.1f}%")
            print(f"   RSI: {row['rsi']:.1f} | ATR: {row['ATR']:.2f}")
        else:  # SELL
            print(f"🔴 SIGNAL: SELL")
            print(f"   Time: {row['Datetime']}")
            print(f"   Entry: {row['Close']:.2f}")
            print(f"   Stop Loss: {row['sl_sell']:.2f}")
            print(f"   Take Profit: {row['tp_sell']:.2f}")
            print(f"   Confidence: {row['sell_proba']*100:.1f}%")
            print(f"   RSI: {row['rsi']:.1f} | ATR: {row['ATR']:.2f}")
        print("-"*70)


latest = df_last_50.iloc[-1]

if latest['signal'] == 'HOLD':
    print(f"\n⚪ NO SIGNAL")
    print(f"Time: {latest['Datetime']}")
    print(f"Current Price: {latest['Close']:.2f}")
    print(f"BUY Probability: {latest['buy_proba']*100:.1f}% (threshold: {buy_threshold*100:.1f}%)")
    print(f"SELL Probability: {latest['sell_proba']*100:.1f}% (threshold: {sell_threshold*100:.1f}%)")
elif latest['signal'] == 'BUY':
    print(f"\n🟢 BUY SIGNAL")
    print(f"Time: {latest['Datetime']}")
    print(f"Entry: {latest['Close']:.2f}")
    print(f"Stop Loss: {latest['sl_buy']:.2f}")
    print(f"Take Profit: {latest['tp_buy']:.2f}")
    print(f"Confidence: {latest['buy_proba']*100:.1f}%")
else:  # SELL
    print(f"\n🔴 SELL SIGNAL")
    print(f"Time: {latest['Datetime']}")
    print(f"Entry: {latest['Close']:.2f}")
    print(f"Stop Loss: {latest['sl_sell']:.2f}")
    print(f"Take Profit: {latest['tp_sell']:.2f}")
    print(f"Confidence: {latest['sell_proba']*100:.1f}%")

print("\n✅ Inference complete!")

✅ Model loaded successfully!
Buy threshold: 0.6000
Sell threshold: 0.6000

📊 Loaded last 300 candles for inference

🔧 Creating features...
After feature engineering: 101 rows

🎯 Generating signals for last 50 rows...

📊 SIGNALS FOR LAST 50 CANDLES

Found 11 trading signals:

🔴 SIGNAL: SELL
   Time: 2025-10-03 01:00:00
   Entry: 3878.30
   Stop Loss: 3896.91
   Take Profit: 3841.08
   Confidence: 60.4%
   RSI: 48.1 | ATR: 12.41
----------------------------------------------------------------------
🔴 SIGNAL: SELL
   Time: 2025-10-03 02:00:00
   Entry: 3882.00
   Stop Loss: 3900.35
   Take Profit: 3845.29
   Confidence: 71.1%
   RSI: 50.3 | ATR: 12.24
----------------------------------------------------------------------
🔴 SIGNAL: SELL
   Time: 2025-10-03 03:00:00
   Entry: 3887.30
   Stop Loss: 3905.28
   Take Profit: 3851.35
   Confidence: 63.5%
   RSI: 53.5 | ATR: 11.98
----------------------------------------------------------------------
🔴 SIGNAL: SELL
   Time: 2025-10-03 04:00:00
  